# DeepBaseEditor efficiency: fixed split from TXT inputs

Inputs: `target_context_30nt.txt`, `activity.txt`, optional `chromatin_accessibility.txt`, and optional `group.txt`.

Set `EDITOR_MODE` to `ABE_Efficiency`, `CBE_Efficiency`, or `CBE_Efficiency_CA`. The split uses seed 42, a 12,000-sample seen pool, an 80/20 train/validation split, and all remaining samples as unseen.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

EDITOR_MODE = "CBE_Efficiency"
# EDITOR_MODE = "CBE_Efficiency"
# EDITOR_MODE = "CBE_Efficiency_CA"

SEQUENCE_FILE = Path("target_context_30nt.txt")
ACTIVITY_FILE = Path("activity.txt")
CHROMATIN_FILE = Path("chromatin_accessibility.txt")
GROUP_FILE = Path("group.txt")

SPLIT_SEED = 42
SELECTED_SEEN_SIZE = 4154
VALIDATION_FRACTION = 0.20

BASE_OUTPUT_DIR = Path(f"results/deepbaseeditor_{EDITOR_MODE.lower()}_fixed_split")
SPLIT_DIR = BASE_OUTPUT_DIR / "saved_splits"
TXT_DIR = BASE_OUTPUT_DIR / "split_sequences"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TXT_DIR.mkdir(parents=True, exist_ok=True)

for p in [SEQUENCE_FILE, ACTIVITY_FILE]:
    assert p.exists(), f"Missing file: {p.resolve()}"

print("Editor mode:", EDITOR_MODE)
print("Output:", BASE_OUTPUT_DIR.resolve())


In [ ]:
def read_lines(path):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]

sequences = [x.upper() for x in read_lines(SEQUENCE_FILE)]
activities = [float(x) for x in read_lines(ACTIVITY_FILE)]

if len(sequences) != len(activities):
    raise ValueError("Sequence and activity counts do not match.")

bad = [(i, s) for i, s in enumerate(sequences)
       if len(s) != 30 or not set(s).issubset({"A","C","G","T"})]
if bad:
    raise ValueError(f"Invalid 30-nt sequences: {bad[:10]}")

data = pd.DataFrame({
    "target_context_30nt": sequences,
    "activity": activities,
})

if CHROMATIN_FILE.exists():
    chromatin = [float(x) for x in read_lines(CHROMATIN_FILE)]
    if len(chromatin) != len(data):
        raise ValueError("Chromatin count does not match.")
    if not set(np.unique(chromatin)).issubset({0.0, 1.0}):
        raise ValueError("Chromatin values must be 0/1.")
    data["chromatin_accessibility"] = chromatin

if GROUP_FILE.exists():
    groups = read_lines(GROUP_FILE)
    if len(groups) != len(data):
        raise ValueError("Group count does not match.")
    data["group"] = groups

if EDITOR_MODE == "CBE_Efficiency_CA" and "chromatin_accessibility" not in data.columns:
    raise ValueError("CBE_Efficiency_CA requires chromatin_accessibility.txt")

data["sample_id"] = np.arange(len(data))

if len(data) <= SELECTED_SEEN_SIZE:
    raise ValueError(
        f"Need more than {SELECTED_SEEN_SIZE} samples; found {len(data)}."
    )

display(data.head())
print("Samples:", len(data))


In [ ]:
np.random.seed(SPLIT_SEED)
full_indices = np.arange(len(data))
selected_indices = np.random.choice(
    len(full_indices), size=SELECTED_SEEN_SIZE, replace=False
)
unseen_indices = np.setdiff1d(full_indices, selected_indices)
train_indices, validation_indices = train_test_split(
    selected_indices,
    test_size=VALIDATION_FRACTION,
    random_state=SPLIT_SEED,
)

train_data = data.iloc[train_indices].reset_index(drop=True)
validation_data = data.iloc[validation_indices].reset_index(drop=True)
unseen_data = data.iloc[unseen_indices].reset_index(drop=True)

display(pd.DataFrame({
    "subset": ["train", "validation", "unseen"],
    "n_rows": [len(train_data), len(validation_data), len(unseen_data)],
}))


In [ ]:
train_file = SPLIT_DIR / "train_split.csv"
validation_file = SPLIT_DIR / "validation_split.csv"
unseen_file = SPLIT_DIR / "unseen_split.csv"

train_data.to_csv(train_file, index=False)
validation_data.to_csv(validation_file, index=False)
unseen_data.to_csv(unseen_file, index=False)

np.savetxt(BASE_OUTPUT_DIR / "train_indices.txt", train_indices, fmt="%d")
np.savetxt(BASE_OUTPUT_DIR / "validation_indices.txt", validation_indices, fmt="%d")
np.savetxt(BASE_OUTPUT_DIR / "unseen_indices.txt", unseen_indices, fmt="%d")

def export_subset(df, name):
    out = df.copy()
    out["model_input_24nt"] = out["target_context_30nt"].str.slice(0, 24)
    out["upstream_4nt"] = out["target_context_30nt"].str.slice(0, 4)
    out["protospacer_20nt"] = out["target_context_30nt"].str.slice(4, 24)
    out["pam_3nt"] = out["target_context_30nt"].str.slice(24, 27)
    out["downstream_3nt"] = out["target_context_30nt"].str.slice(27, 30)

    d = TXT_DIR / name
    d.mkdir(parents=True, exist_ok=True)

    cols = [
        "sample_id", "target_context_30nt", "model_input_24nt",
        "upstream_4nt", "protospacer_20nt", "pam_3nt",
        "downstream_3nt", "activity"
    ]
    if "chromatin_accessibility" in out.columns:
        cols.append("chromatin_accessibility")
    if "group" in out.columns:
        cols.append("group")

    for col in cols:
        out[col].to_csv(d / f"{name}_{col}.txt", index=False, header=False)

    out[cols].to_csv(d / f"{name}_complete.tsv", sep="\t", index=False)

export_subset(train_data, "train")
export_subset(validation_data, "validation")
export_subset(unseen_data, "unseen")


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "editor_mode": EDITOR_MODE,
    "split_seed": SPLIT_SEED,
    "selected_seen_size": SELECTED_SEEN_SIZE,
    "validation_fraction_within_seen": VALIDATION_FRACTION,
    "n_total": int(len(data)),
    "n_train": int(len(train_data)),
    "n_validation": int(len(validation_data)),
    "n_unseen": int(len(unseen_data)),
    "train_sha256": sha256(train_file),
    "validation_sha256": sha256(validation_file),
    "unseen_sha256": sha256(unseen_file),
}

with open(BASE_OUTPUT_DIR / "split_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved to:", BASE_OUTPUT_DIR.resolve())
